# Decision trees
**Machine Learning · Week 3 (Classification) · Day 4**

Lesson page: https://sara-aminpour.com/tutorials/ml/w3d4.html

A decision tree classifies by asking yes/no questions about one feature at a time.
Today: how it picks each question, and how deep is too deep.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.tree import DecisionTreeClassifier, plot_tree

X, y = make_moons(n_samples=300, noise=0.25, random_state=3)
tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X, y)
print("training accuracy of a depth-3 tree:", round(tree.score(X, y), 3))

plt.figure(figsize=(9, 4))
plot_tree(tree, feature_names=["x1", "x2"], class_names=["orange", "blue"], filled=True, rounded=True, fontsize=8)
plt.show()

Each box is a question like `x2 <= 0.25`; going left means yes. `gini` is the impurity of the examples that
reach the box, `samples` how many there are, and `value` how many of each class.

## The Gini impurity by hand
For a group with a fraction $p$ of class 1, Gini $= 1 - p^2 - (1-p)^2$. A split is scored by the
**weighted** Gini of its two halves; the tree keeps the question with the lowest score.

In [ ]:
def gini(labels):
    if len(labels) == 0:
        return 0.0
    p = np.mean(labels)
    return 1 - p ** 2 - (1 - p) ** 2

root = tree.tree_
t, f = root.threshold[0], root.feature[0]
left, right = y[X[:, f] <= t], y[X[:, f] > t]
print(f"first question: x{f + 1} <= {t:.3f}")
print(f"before: Gini {gini(y):.3f} (all {len(y)} examples)")
print(f"after:  left {len(left)} examples, Gini {gini(left):.3f}; right {len(right)}, Gini {gini(right):.3f}")
print(f"weighted Gini = {(len(left) * gini(left) + len(right) * gini(right)) / len(y):.3f}  (scikit-learn: {root.impurity[0]:.3f} before the split)")

## How deep is too deep?

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
depths = range(1, 16)
train_acc, test_acc = [], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_tr, y_tr)
    train_acc.append(m.score(X_tr, y_tr))
    test_acc.append(m.score(X_te, y_te))
plt.figure(figsize=(6, 3.2))
plt.plot(depths, train_acc, "o-", color="#17a98c", label="training")
plt.plot(depths, test_acc, "o-", color="#7b5cfa", label="test")
plt.xlabel("max_depth")
plt.ylabel("accuracy")
plt.legend()
plt.title("Deep trees memorize their training data")
plt.show()

## Which features did the tree use?
On the breast-cancer data (30 measurements), a tree tells you which features its questions rely on.

In [ ]:
from sklearn.datasets import load_breast_cancer

bc = load_breast_cancer()
m = DecisionTreeClassifier(max_depth=4, random_state=0).fit(bc.data, bc.target)
order = np.argsort(-m.feature_importances_)[:5]
for i in order:
    print(f"{bc.feature_names[i]:>24}: importance {m.feature_importances_[i]:.2f}")

## Your turn: find the best question yourself
Write `best_split(x, labels)` for **one** feature: try a threshold halfway between every pair of neighboring
sorted values, score each with the weighted Gini, and return the best threshold. Compare with a
scikit-learn "stump" (a tree of depth 1) on the first feature of the moons.

In [ ]:
def best_split(x, labels):
    values = np.unique(x)
    best_t, best_score = None, np.inf
    for a, b in zip(values[:-1], values[1:]):
        t = (a + b) / 2
        # TODO: weighted Gini of labels[x <= t] and labels[x > t]; keep the lowest
    return best_t, best_score

print("mine:   ", best_split(X[:, 0], y))
stump = DecisionTreeClassifier(max_depth=1).fit(X[:, [0]], y)
print("sklearn:", stump.tree_.threshold[0])

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def best_split(x, labels):
    values = np.unique(x)
    best_t, best_score = None, np.inf
    for a, b in zip(values[:-1], values[1:]):
        t = (a + b) / 2
        L, R = labels[x <= t], labels[x > t]
        score = (len(L) * gini(L) + len(R) * gini(R)) / len(labels)
        if score < best_score:
            best_t, best_score = t, score
    return best_t, best_score

t, s = best_split(X[:, 0], y)
print(f"mine:    threshold {t:.4f}, weighted Gini {s:.4f}")
print(f"sklearn: threshold {stump.tree_.threshold[0]:.4f}")
print("A whole tree repeats this search for every feature, in every box, level after level.")